In [1]:
import numpy as np
import pandas as pd

bls_workbook = pd.ExcelFile('../data/raw/bls/occupation.xlsx')
print(bls_workbook.sheet_names)

for sheet in bls_workbook.sheet_names:
    preview = pd.read_excel(bls_workbook, sheet, header=None, nrows=5)
    print('=====', sheet, preview.shape)
    print(preview.iloc[:5, :10].to_string(), '\n')

['Index', 'Table 1.1', 'Table 1.2', 'Table 1.3', 'Table 1.4', 'Table 1.5', 'Table 1.6', 'Table 1.8', 'Table 1.9', 'Table 1.10', 'Table 1.11', 'Table 1.12']
===== Index (5, 1)
                                                               0
0                                       Occupation.xlsx contents
1               Table 1.1 Employment by major occupational group
2  Table 1.2 Occupational projections and worker characteristics
3                          Table 1.3 Fastest growing occupations
4                 Table 1.4 Occupations with the most job growth 

===== Table 1.1 (5, 7)
                                                                                                     0                                     1                 2                 3                                    4                                    5                                  6
0  Table 1.1 Employment by major occupational group, 2025 and projected 2035 (Employment in thousands)                      

In [2]:
census_to_onet = pd.read_parquet('../data/derived/census-onet-crosswalk.parquet')
job_exposure = pd.read_csv('../data/raw/anthropic/job_exposure.csv')
print(job_exposure.shape)
print(job_exposure['observed_exposure'].describe().round(3), '\n')

# census code -> 6-digit SOC (one row per pair; several O*NET codes can share a SOC)
census_to_soc = (census_to_onet.assign(soc=census_to_onet['onet'].str[:7])
                 [['census', 'soc']].drop_duplicates())

census_exposure_pairs = census_to_soc.merge(job_exposure, left_on='soc', right_on='occ_code', how='left')

# coverage: which SOC codes have no exposure score?
missing_soc = census_exposure_pairs[census_exposure_pairs['observed_exposure'].isna()]['soc'].unique()
print(len(missing_soc), 'SOC codes in the crosswalk have no exposure score:', sorted(missing_soc)[:20])

census_exposure = (census_exposure_pairs.groupby('census')['observed_exposure']
                   .mean().rename('ai_exposure').reset_index())
print(census_exposure['ai_exposure'].notna().sum(), 'of', len(census_exposure), 'census codes have an exposure score\n')

# sanity check: most and least exposed census occupations
census_titles = (pd.read_parquet('../data/derived/census-soc2010-crosswalk.parquet')
                 [['census', 'title']].drop_duplicates('census'))
check = census_exposure.merge(census_titles, on='census').dropna().sort_values('ai_exposure')
print('least exposed:\n', check.head(8).to_string(index=False), '\n')
print('most exposed:\n', check.tail(8).to_string(index=False))

census_exposure.to_parquet('../data/derived/census-ai-exposure.parquet', index=False)

(756, 3)
count    756.000
mean       0.077
std        0.133
min        0.000
25%        0.000
50%        0.000
75%        0.096
max        0.745
Name: observed_exposure, dtype: float64 

37 SOC codes in the crosswalk have no exposure score: ['11-2033', '11-9072', '13-1021', '13-1022', '13-1023', '13-1082', '13-2022', '13-2023', '21-1011', '21-1014', '25-2055', '25-2056', '25-3031', '25-9042', '25-9043', '27-2091', '29-1214', '29-2011', '29-2012', '29-2036']
503 of 517 census codes have an exposure score

least exposed:
  census  ai_exposure                                                                               title
   9740          0.0                                                   Tank car, truck, and ship loaders
   3630          0.0                                                                  Massage therapists
   3640          0.0                                                                   Dental assistants
   7160          0.0                                  

In [5]:
nem = pd.read_excel(bls_workbook, 'Table 1.2', header=1)
print(list(nem.columns), '\n')

def find_col(fragment):
    hits = [c for c in nem.columns if fragment.lower() in str(c).lower()]
    assert len(hits) == 1, (fragment, hits)
    return hits[0]

nem = nem[nem[find_col('Occupation type')] == 'Line item'].rename(columns={
    find_col('Matrix code'): 'nem_code',
    find_col('Matrix title'): 'nem_title',
    find_col('Employment, 2025'): 'emp_2025',
    find_col('Employment, 2035'): 'emp_2035',
    find_col('Median annual wage'): 'median_wage',
    find_col('education'): 'entry_education',
    find_col('Work experience'): 'work_experience',
    find_col('on-the-job'): 'on_the_job_training',
})[['nem_code', 'nem_title', 'emp_2025', 'emp_2035', 'median_wage',
    'entry_education', 'work_experience', 'on_the_job_training']]
wage_text = nem['median_wage'].astype(str).str.replace(r'[^\d.]', '', regex=True)
nem['median_wage'] = pd.to_numeric(wage_text, errors='coerce')
nem[['work_experience', 'on_the_job_training']] = nem[['work_experience', 'on_the_job_training']].fillna('None')
print(nem.shape, 'detailed occupations\n')
print(nem['entry_education'].value_counts(), '\n')

# SOC -> BLS row: exact code first, otherwise the most specific combined code (trailing zeros = prefix)
nem['prefix'] = nem['nem_code'].str.rstrip('0').str.rstrip('-')
def match_nem(soc):
    exact = nem[nem['nem_code'] == soc]
    if len(exact):
        return exact.index[0]
    hits = nem[[soc.startswith(p) for p in nem['prefix']]]
    return hits['prefix'].str.len().idxmax() if len(hits) else None

soc_to_nem = pd.DataFrame({'soc': census_to_soc['soc'].unique()})
soc_to_nem['nem_row'] = soc_to_nem['soc'].map(match_nem)
print(soc_to_nem['nem_row'].isna().sum(), 'SOC codes with no BLS row:',
      sorted(soc_to_nem.loc[soc_to_nem['nem_row'].isna(), 'soc'])[:20], '\n')

pairs = (census_to_soc.merge(soc_to_nem, on='soc')
         .merge(nem.drop(columns='prefix'), left_on='nem_row', right_index=True, how='left')
         .merge(job_exposure[['occ_code', 'observed_exposure']], left_on='soc', right_on='occ_code', how='left')
         .drop_duplicates(['census', 'nem_row', 'soc']))

def summarise(g):
    g = g.dropna(subset=['emp_2025'])
    if g.empty:
        return pd.Series(np.nan, index=['emp_2025', 'growth_pct', 'median_wage', 'entry_education', 'work_experience', 'on_the_job_training', 'ai_exposure'])
    weight = g.drop_duplicates('nem_row')
    exp = g.dropna(subset=['observed_exposure'])
    biggest = weight.loc[weight['emp_2025'].idxmax()]
    return pd.Series({
        'emp_2025': weight['emp_2025'].sum(),
        'growth_pct': 100 * (weight['emp_2035'].sum() / weight['emp_2025'].sum() - 1),
        'median_wage': np.average(weight['median_wage'].fillna(weight['median_wage'].mean()), weights=weight['emp_2025'])
                       if weight['median_wage'].notna().any() else np.nan,
        'entry_education': biggest['entry_education'],
        'work_experience': biggest['work_experience'],
        'on_the_job_training': biggest['on_the_job_training'],
        'ai_exposure': np.average(exp['observed_exposure'], weights=exp['emp_2025']) if len(exp) else np.nan,
    })

census_outlook = pairs.groupby('census').apply(summarise).reset_index()
print(census_outlook.shape)
print(census_outlook.isna().sum(), '\n')

check = census_outlook.merge(census_titles, on='census')
print('fastest growing:\n', check.nlargest(8, 'growth_pct')[['title', 'growth_pct', 'median_wage', 'entry_education']].to_string(index=False), '\n')
print('fastest shrinking:\n', check.nsmallest(8, 'growth_pct')[['title', 'growth_pct', 'median_wage', 'entry_education']].to_string(index=False))

census_outlook.to_parquet('../data/derived/census-outlook.parquet', index=False)
print(census_outlook[['ai_exposure', 'growth_pct']].corr(method='spearman').round(3))

['2025 National Employment Matrix title', '2025 National Employment Matrix code', 'Occupation type', 'Employment, 2025', 'Employment, 2035', 'Employment distribution, percent, 2025', 'Employment distribution, percent, 2035', 'Employment change, numeric, 2025–35', 'Employment change, percent, 2025–35', 'Percent self-employed, 2025', 'Occupational openings, 2025–35 annual average', 'Median annual wage, dollars, 2025', 'Typical education needed for entry', 'Work experience in a related occupation', 'Typical on-the-job training needed to attain competency in the occupation', 'Related Occupational Outlook Handbook (OOH) content', 'Related Occupational Outlook Handbook (OOH) content.1'] 

(831, 8) detailed occupations

entry_education
High school diploma or equivalent    326
Bachelor's degree                    177
No formal educational credential     109
Doctoral or professional degree       73
Postsecondary nondegree award         51
Associate's degree                    48
Master's degree

In [6]:
raw = pd.read_excel(bls_workbook, 'Table 1.2', header=1, dtype=str)
wage_col = 'Median annual wage, dollars, 2025'
code_col = '2025 National Employment Matrix code'
missing_codes = nem.loc[nem['median_wage'].isna(), 'nem_code']
print(raw.loc[raw[code_col].isin(missing_codes), [code_col, '2025 National Employment Matrix title', wage_col]].to_string(index=False))

2025 National Employment Matrix code                                      2025 National Employment Matrix title Median annual wage, dollars, 2025
                             27-2011                                                                     Actors                                 —
                             27-2031                                                                    Dancers                                 —
                             27-2042                                                      Musicians and singers                                 —
                             27-2091                                                 Disc jockeys, except radio                                 —
                             27-2099         Entertainers and performers, sports and related workers, all other                                 —
                             45-3031                                                Fishing and hunting workers             

In [7]:
exposed = census_outlook[census_outlook['ai_exposure'] > 0]
print(len(exposed), 'occupations with some exposure')
print(exposed[['ai_exposure', 'growth_pct']].corr(method='spearman').round(3))

256 occupations with some exposure
             ai_exposure  growth_pct
ai_exposure        1.000      -0.148
growth_pct        -0.148       1.000


In [8]:
from scipy.stats import spearmanr
print(spearmanr(exposed['ai_exposure'], exposed['growth_pct'], nan_policy='omit'))

SignificanceResult(statistic=np.float64(-0.14798443071091552), pvalue=np.float64(0.017825511995871436))


In [9]:
import pandas as pd
from scipy.stats import spearmanr

eloundou = pd.read_csv('../data/raw/eloundou/occ_level.csv').rename(columns={
    'O*NET-SOC Code': 'onet',
    'Title': 'onet_title',
    'dv_rating_beta': 'theoretical_gpt4',
    'human_rating_beta': 'theoretical_human',
})

census_onet = pd.read_parquet('../data/derived/census-onet-crosswalk.parquet')
matched = census_onet.merge(eloundou[['onet', 'onet_title', 'theoretical_gpt4', 'theoretical_human']],
                            on='onet', how='left')
print('O*NET codes matched:', matched['theoretical_gpt4'].notna().sum(), 'of', len(matched))

theoretical = matched.groupby('census').agg(
    title=('onet_title', 'first'),
    theoretical_gpt4=('theoretical_gpt4', 'mean'),
    theoretical_human=('theoretical_human', 'mean'),
).reset_index()
print('census codes covered:', theoretical['theoretical_gpt4'].notna().sum(), 'of', len(theoretical))

outlook = pd.read_parquet('../data/derived/census-outlook.parquet')
if 'census' not in outlook.columns:
    outlook = outlook.reset_index()
exposure = theoretical.merge(outlook[['census', 'ai_exposure']], on='census', how='left')

both = exposure.dropna(subset=['theoretical_gpt4', 'ai_exposure'])
print('\nSpearman, GPT-4 vs human rating:', round(spearmanr(both['theoretical_gpt4'], both['theoretical_human'])[0], 3))
print('Spearman, theoretical vs observed (all):', spearmanr(both['theoretical_gpt4'], both['ai_exposure']))
observed_nonzero = both[both['ai_exposure'] > 0]
print('Spearman, theoretical vs observed (observed > 0):', spearmanr(observed_nonzero['theoretical_gpt4'], observed_nonzero['ai_exposure']))

# Different scales, so compare percentile ranks: positive gap = exposed in theory, not (yet) in usage
exposure['theoretical_pct'] = exposure['theoretical_gpt4'].rank(pct=True)
exposure['observed_pct'] = exposure['ai_exposure'].rank(pct=True)
exposure['exposure_gap'] = exposure['theoretical_pct'] - exposure['observed_pct']

show = ['census', 'title', 'theoretical_gpt4', 'ai_exposure', 'exposure_gap']
print('\nExposed in theory, low observed usage:')
print(exposure.nlargest(15, 'exposure_gap')[show].round(2).to_string(index=False))
print('\nObserved usage ahead of theory:')
print(exposure.nsmallest(10, 'exposure_gap')[show].round(2).to_string(index=False))

exposure.to_parquet('../data/derived/census-theoretical-exposure.parquet', index=False)

O*NET codes matched: 940 of 940
census codes covered: 517 of 517

Spearman, GPT-4 vs human rating: 0.92
Spearman, theoretical vs observed (all): SignificanceResult(statistic=np.float64(0.7379133929639484), pvalue=np.float64(1.3506907977263442e-87))
Spearman, theoretical vs observed (observed > 0): SignificanceResult(statistic=np.float64(0.6463290660905514), pvalue=np.float64(1.1377385301939823e-31))

Exposed in theory, low observed usage:
 census                                                              title  theoretical_gpt4  ai_exposure  exposure_gap
   5020                                                Telephone Operators              0.89          0.0          0.74
    330                                                  Gambling Managers              0.57          0.0          0.62
   5340                                                New Accounts Clerks              0.56          0.0          0.61
   1510                                                  Nuclear Engineers   

In [10]:
outlook = pd.read_parquet('../data/derived/census-outlook.parquet')
if 'census' not in outlook.columns:
    outlook = outlook.reset_index()
growth_check = exposure.merge(outlook[['census', 'growth_pct']], on='census').dropna(subset=['theoretical_gpt4', 'growth_pct'])

print('n =', len(growth_check))
print('Spearman, theoretical exposure vs projected growth:', spearmanr(growth_check['theoretical_gpt4'], growth_check['growth_pct']))
print('Spearman, human-rated version:', spearmanr(growth_check['theoretical_human'], growth_check['growth_pct']))

n = 516
Spearman, theoretical exposure vs projected growth: SignificanceResult(statistic=np.float64(0.07907728114559444), pvalue=np.float64(0.07269425551940968))
Spearman, human-rated version: SignificanceResult(statistic=np.float64(0.13271399937794146), pvalue=np.float64(0.0025215475670217727))


In [11]:
import numpy as np
from scipy.stats import spearmanr, pearsonr

job_zones = pd.read_csv('../data/raw/db_31_0_csv/job_zones.csv').rename(columns={'O*NET-SOC Code': 'onet', 'Job Zone': 'job_zone'})
census_onet = pd.read_parquet('../data/derived/census-onet-crosswalk.parquet')
census_zone = (census_onet.merge(job_zones[['onet', 'job_zone']], on='onet', how='left')
               .groupby('census')['job_zone'].median().round().reset_index())

zone_check = growth_check.merge(census_zone, on='census', how='left').dropna(subset=['job_zone'])
zone_check['job_zone'] = zone_check['job_zone'].astype(int)
print('n =', len(zone_check))

print('\nDoes job zone drive both?')
print('  zone vs exposure:', round(spearmanr(zone_check['job_zone'], zone_check['theoretical_gpt4'])[0], 3))
print('  zone vs growth:  ', round(spearmanr(zone_check['job_zone'], zone_check['growth_pct'])[0], 3))

print('\nWithin each job zone:')
for zone, group in zone_check.groupby('job_zone'):
    rho, p = spearmanr(group['theoretical_gpt4'], group['growth_pct'])
    print(f'  zone {zone}: n = {len(group):3d}  rho = {rho:+.3f}  p = {p:.3f}')

# Partial Spearman: rank both variables, subtract each zone's average rank, correlate what's left
exposure_rank = zone_check['theoretical_gpt4'].rank()
growth_rank = zone_check['growth_pct'].rank()
exposure_within = exposure_rank - exposure_rank.groupby(zone_check['job_zone']).transform('mean')
growth_within = growth_rank - growth_rank.groupby(zone_check['job_zone']).transform('mean')
partial_rho, partial_p = pearsonr(exposure_within, growth_within)
overall_rho = spearmanr(zone_check['theoretical_gpt4'], zone_check['growth_pct'])[0]
print(f'\nControlling for job zone: rho = {partial_rho:+.3f}  p = {partial_p:.4f}  (was {overall_rho:+.3f} without)')

n = 516

Does job zone drive both?
  zone vs exposure: 0.568
  zone vs growth:   0.411

Within each job zone:
  zone 2: n = 249  rho = -0.077  p = 0.227
  zone 3: n = 104  rho = -0.404  p = 0.000
  zone 4: n = 120  rho = -0.119  p = 0.195
  zone 5: n =  43  rho = -0.153  p = 0.326

Controlling for job zone: rho = -0.208  p = 0.0000  (was +0.079 without)


In [12]:
# BLS relative AI exposure (supplement to the 2025–35 projections): one category per BLS occupation.
# Census codes covering several BLS occupations get the employment-weighted average, rounded to a category.
bls_ai = pd.read_excel('../data/raw/bls/ai-exposure-categories.xlsx', 'AI Exposure Categories', header=1)
bls_ai = bls_ai.rename(columns={
    '2025 National Employment Matrix code': 'nem_code',
    'Relative AI exposure': 'bls_ai_category',
})[['nem_code', 'bls_ai_category']].dropna()

category_rank = {'Low': 1, 'Moderate': 2, 'High': 3, 'Very high': 4}
rank_category = {rank: name for name, rank in category_rank.items()}
bls_ai['bls_ai_rank'] = bls_ai['bls_ai_category'].map(category_rank)
print(bls_ai['bls_ai_category'].value_counts(), '\n')

census_nem = (pairs.drop_duplicates(['census', 'nem_code'])
              .merge(bls_ai, on='nem_code', how='inner')
              .dropna(subset=['emp_2025']))

census_bls = census_nem.groupby('census').apply(
    lambda g: np.average(g['bls_ai_rank'], weights=g['emp_2025']), include_groups=False
).rename('bls_ai_rank').reset_index()
census_bls['bls_ai_category'] = census_bls['bls_ai_rank'].round().astype(int).map(rank_category)
print('census codes covered:', len(census_bls), '\n')
print(census_bls['bls_ai_category'].value_counts(), '\n')

# How well does it agree with the Eloundou score the app shows now?
check = census_bls.merge(exposure[['census', 'title', 'theoretical_gpt4']], on='census')
print('Spearman, BLS rank vs Eloundou:', spearmanr(check['bls_ai_rank'], check['theoretical_gpt4']))
print(check[check['census'].isin([1030, 1020, 1240])].to_string(index=False))

census_bls.to_parquet('../data/derived/census-bls-exposure.parquet', index=False)

bls_ai_category
Low          213
Very high    206
High         206
Moderate     206
Name: count, dtype: int64 

census codes covered: 516 

bls_ai_category
Low          144
Very high    134
Moderate     129
High         109
Name: count, dtype: int64 

Spearman, BLS rank vs Eloundou: SignificanceResult(statistic=np.float64(0.8986740479347549), pvalue=np.float64(4.211331196774137e-186))
 census  bls_ai_rank bls_ai_category                      title  theoretical_gpt4
   1020          4.0       Very high        Software Developers          0.872807
   1030          4.0       Very high             Web Developers          0.810897
   1240          4.0       Very high Bioinformatics Technicians          0.737784
